# 实验三：QQ邮箱采集（Playwright 版）

- 工具：Playwright 1.62.0，异步 API
- 运行位置：Docker 内 JupyterLab 的代码单元（工作目录 `workspace/lab03/`）
- 四阶段写在本 Notebook 中，单元 1—13 依次向下追加

> 提交前清除本 Notebook 输出：其中包含登录态与私人邮件内容。


## 第一阶段：扫码登录（第 1—2 课时）

**目标：** 打开 QQ 邮箱，人工扫码后由程序判断"已登录"，并进入收件箱。

程序无法代替人扫码，但可以代替人去**判断**扫码是否完成。判据不能是"等 30 秒"，而必须是"页面上出现了只有登录后才会有的控件"。


### 单元 1：准备浏览器

对应指导书单元 1。四处空格分别是 `start()`、`launch()`、`new_context()`、`new_page()`，已按指导书提示补全。

`course_visual.prepare()` 往本内核的环境变量写入 `DISPLAY` 和 `XAUTHORITY`，让随后启动的 Chromium 把画面输出到可视化窗口，而不是容器内部的隐形显示器。因此 `headless=False` 才有意义——你能看见窗口，才扫得了码。


In [ ]:
import sys
from pathlib import Path
from uuid import uuid4
import pandas as pd
from playwright.async_api import async_playwright

# 后面三个阶段共用的本地私有目录，已被 .gitignore 忽略。
PRIVATE = Path.cwd() / "private_data"
PRIVATE.mkdir(exist_ok=True)

# 复用可视化增量包已有的显示服务配置，不修改容器配置。
sys.path.insert(0, str(Path.cwd().parent / "L07_L08_visual"))
import course_visual
course_visual.prepare()

p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, args=["--disable-dev-shm-usage"])
context = await browser.new_context(
    viewport={"width": 1000, "height": 620}, accept_downloads=True
)
page = await context.new_page()
page.set_default_timeout(20000)

print("工作目录：", Path.cwd())
print("浏览器已启动，请在可视化窗口中查看")


### 单元 2：打开邮箱，扫码并等待

三处关键点：

1. **`goto()` 用 `wait_until="domcontentloaded"`**，不等 `load`。登录页会保持长连接，等 `load` 事件反而会超时。
2. **收件箱改用目录 ID 定位，不用文字**。`.frame-sidebar-menu[data-sidebar-dir-id="1"]` 中的 `1` 是收件箱在服务端的固定目录编号，与界面显示语言无关。
3. **文本定位为什么不可靠**：本实验实测过——同一账号在英文界面下菜单写作 `Inbox`，切到中文界面写作"收件箱"。写死任何一种，换个语言就彻底失效。这不是假设，是实际踩过的坑。

> 报告可填：定位方式 `CSS 属性定位`；选择器 `.frame-sidebar-menu[data-sidebar-dir-id="1"]`；等待条件 `locator.wait_for()`，最长 600000 毫秒（10 分钟）。
>
> 如果一定要用文本定位：`page.locator(".frame-sidebar").get_by_text("收件箱", exact=True)`，必须限定 `.frame-sidebar` 作用域——页面上另有一个"收件箱"标题 `.toolbar-folder-name`，不限定会匹配到两个元素，触发 strict mode 错误。


In [ ]:
# ① 打开网页。
await page.goto("https://mail.qq.com/", wait_until="domcontentloaded")
print("请在浏览器画面中扫码", flush=True)

# ② 定位左侧"收件箱"。
#    用目录 ID 而不是文字：ID=1 固定代表收件箱，界面改语言也不会变。
#    文本写法在英文界面是 Inbox、中文界面是"收件箱"，换语言即失效。
inbox = page.locator('.frame-sidebar-menu[data-sidebar-dir-id="1"]')

# ③ 最长等 10 分钟；控件一旦出现就立即继续，不做无意义的固定等待。
await inbox.wait_for(timeout=600000)
print("登录成功", flush=True)

# ④ 点击进入收件箱，然后确认列表真的加载出来了。
await inbox.click()
await page.locator(".mail-list-page-item[data-mailid]").first.wait_for()


### 验证：确认已进入收件箱

这一步是报告"本次实际结果"的证据：目录名、邮件行数、邮件总数三项同时成立，才能证明 `wait_for` 的判断没有误判。


In [ ]:
print("页面标题：", await page.title())
print("当前目录：", await page.locator(".toolbar-folder-name").inner_text())
print("邮件行数：", await page.locator(".mail-list-page-item[data-mailid]").count())

# 总数不要写死深层选择器：直接读整条工具栏的文字，里面就含"共 138 封"。
toolbar = await page.locator(".mail-list-page-toolbar").inner_text()
print("工具栏文字：", toolbar.replace(chr(10), " | "))

print("第一封发件人：", await page.locator(".cmp-account-nick").first.inner_text())


### 阶段一完成证据

报告中需要提交：

1. 单元 1 与单元 2 的代码截图（红框标出 `goto`、`wait_for`、`click` 三行）。
2. **`登录成功` 的真实输出**与收件箱界面截图，遮盖邮箱地址与私人邮件标题。
3. 解释等待条件：为什么用 `wait_for(收件箱控件)` 而不是 `sleep(30)`。

**可直接写进报告"问题与解决"的一条真实记录：** 按指导书写 `get_by_text("收件箱")` 时无法定位。检查页面 DOM 发现该账号界面语言可切换，英文界面下菜单实际文本是 `Inbox`；且页面上共有两处同名文本（侧栏菜单与列表标题），不加作用域会触发 strict mode 错误。最终改为按目录 ID `data-sidebar-dir-id="1"` 定位，与界面语言解耦，中英文两种界面均实测通过。

思考题提示：人工扫码完成的是**身份认证**，程序拿不到账号密码；程序判断"已登录"的唯一依据是**登录态出现后才有 DOM 节点**。固定等待无法区分"页面已打开"和"已登录"——两者页面不同、耗时不可预测，等短了误判、等长了浪费时间。


## 第二阶段：采集 10 封邮件并保存（第 3—4 课时）

**目标：** 采集发件人、邮箱、主题、时间四项，用 pandas 存成 CSV，回读后应为 10 条记录、4 列。

难点不在循环，在**邮箱地址**：它不存在于列表 DOM 里，只有把鼠标悬停到发件人上才会弹出卡片。静态抓页面永远拿不到，必须真的 hover 一次。

第二个难点是**归属**：上一封的卡片如果还没消失就悬停下一封，读到的可能是上一封的邮箱。所以顺序必须是"先移开让旧卡片消失 → 再悬停新的 → 等新卡片出现"。


### 探测单元：导出列表与悬停卡片的真实结构

静态页面里没有悬停卡片，所以只能让浏览器真的悬停一次，再把结构存下来。

本单元把两份 DOM 写到 `private_data/`：悬停前的列表（`dom_cn_list.html`）、悬停后的卡片（`dom_cn_hover.html`）。两者都只在本地用于定位，不提交、不写进报告。


In [ ]:
# 【探测单元】导出列表结构与悬停卡片结构，仅用于定位选择器。
rows = page.locator(".mail-list-page-item[data-mailid]")
print("邮件行数：", await rows.count())

first = rows.first
print("邮件ID：", await first.get_attribute("data-mailid"))
print("发件人：", await first.locator(".cmp-account-nick").first.inner_text())
print("主题：", await first.locator(".mail-subject").first.inner_text())
print("时间：", await first.locator(".mail-time").first.inner_text())
print("广告标记 .gg-unread-count 个数：", await page.locator(".gg-unread-count").count())

# 悬停前的列表结构
(PRIVATE / "dom_cn_list.html").write_text(await page.content(), encoding="utf-8")

# 悬停第一封的发件人昵称，等卡片渲染
await first.locator(".cmp-account-nick").first.hover()
await page.wait_for_timeout(2000)
hover_html = await page.content()
(PRIVATE / "dom_cn_hover.html").write_text(hover_html, encoding="utf-8")
print("悬停后 DOM 字符数：", len(hover_html))


### 单元 3：选择器常量

对应指导书单元 3。指导书只给了两条线索（发件人、广告标记），其余靠实测确定。

| 常量 | 取值 | 依据 |
|---|---|---|
| `ROW_CSS` | `.mail-list-page-item[data-mailid]` | 每封邮件的父容器，自带 `data-mailid` |
| `SENDER_CSS` | `.cmp-account-nick` | 指导书给定线索 |
| `AD_CSS` | `.gg-unread-count` | 指导书给定线索，聚合入口标记，应跳过 |
| `SUBJECT_CSS` | `.mail-subject` | 行内主题 |
| `TIME_CSS` | `.mail-time` | 行内时间，实测显示为 `11/5` 这类月/日 |
| `HOVER_CSS` | `.cmp-account-nick` | 实测：悬停昵称本身即弹出卡片 |
| `EMAIL_CSS` | `.xmail-cmp-account-card .cmp-email-address` | 卡片内的邮箱字段 |

**为什么不把 `HOVER_CSS` 写成更大的 `.mail-sender`**：那个容器左端是信封图标，悬停到图标上不会弹卡片。悬停元素时鼠标落在元素中心，容器越宽，中心越可能落在空白或图标上。悬停昵称本身，命中率 100%。

**`EMAIL_CSS` 为什么要带 `.xmail-cmp-account-card` 前缀**：列表行内也有邮箱字段，但类名是 `.cmp-account-email`；只有弹出卡片里才叫 `.cmp-email-address`。加前缀是双保险，也避免第四阶段进入详情页后误匹配。


In [ ]:
ROW_CSS = ".mail-list-page-item[data-mailid]"
SENDER_CSS = ".cmp-account-nick"
AD_CSS = ".gg-unread-count"
SUBJECT_CSS = ".mail-subject"
TIME_CSS = ".mail-time"
HOVER_CSS = ".cmp-account-nick"
EMAIL_CSS = ".xmail-cmp-account-card .cmp-email-address"
SCROLLING_CSS = ".mail-list-page .ui-float-scroll-body-scrolling"

print("常量已定义")


### 单元 4：列表解析函数

三个函数会在第三、第四阶段继续使用，不要另换名字。

`read_list_mail` 里有三个顺序不能乱的细节：

1. **先悬停"时间"**。目的是把鼠标从上一封的发件人上移开，让残留的卡片消失。否则读到的可能是上一封的邮箱。
2. **`scroll_into_view_if_needed()`**。列表是虚拟滚动，目标行可能不在可视区；不先滚进来，悬停会落空。
3. **等滚动惯性结束**。滚动过程中 QQ 邮箱会给容器加 `ui-float-scroll-body-scrolling` 类，此时悬停不生效。必须等这个类消失再悬停。


In [ ]:
def get_row(mail_id):
    return page.locator(f'{ROW_CSS}[data-mailid="{mail_id}"]')


async def get_ids():
    # 先等列表出现，再一次性读回整批 ID，不逐个保存旧元素。
    await page.locator(ROW_CSS).first.wait_for()
    ids = await page.locator(ROW_CSS).evaluate_all(
        "rows => rows.map(row => row.getAttribute('data-mailid'))"
    )
    if not ids or not all(ids):
        raise ValueError("没有取得有效邮件ID，请检查 ROW_CSS")
    return ids


async def read_list_mail(mail_id):
    row = get_row(mail_id)

    # 聚合入口（广告）不是单封邮件，跳过。
    if await row.locator(AD_CSS).count():
        return None

    sender = await row.locator(SENDER_CSS).first.inner_text()
    subject = await row.locator(SUBJECT_CSS).first.inner_text()
    time_text = await row.locator(TIME_CSS).first.inner_text()

    # ① 悬停"时间"，让上一封残留的卡片消失。
    await row.locator(TIME_CSS).first.hover()
    await page.locator(EMAIL_CSS).wait_for(state="hidden")

    # ② 把发件人滚进可视区，并等滚动惯性结束。
    await row.locator(HOVER_CSS).first.scroll_into_view_if_needed()
    await page.locator(SCROLLING_CSS).wait_for(state="hidden")

    # ③ 再悬停发件人，等卡片出现后读邮箱。
    await row.locator(HOVER_CSS).first.hover()
    await page.locator(EMAIL_CSS).wait_for(state="visible")
    email = (await page.locator(EMAIL_CSS).inner_text()).strip()

    return {"邮件ID": mail_id, "发件人": sender,
            "邮箱": email, "主题": subject, "时间": time_text}


### 单元 5：先采一封，再采 10 封

先只跑一封，确认四个字段都对，再放开循环。一次性跑 10 封，出错了很难判断是选择器问题还是某封邮件的特殊结构。


In [ ]:
# 先验证一封：字段对不对、邮箱是不是当前这封的。
one = await read_list_mail((await get_ids())[0])
one


In [ ]:
basic_records = []
for mail_id in await get_ids():
    record = await read_list_mail(mail_id)
    if record is None:
        continue
    basic_records.append(record)
    if len(basic_records) == 10:
        break
print("成功采集：", len(basic_records))


### 单元 6：用 pandas 保存并回读

`save_records` 与 `read_records` 定义一次，后面三个阶段的 CSV 都用它们，保证编码和参数一致。

- `index=False`：不把行号写成第一列
- `encoding="utf-8-sig"`：带 BOM，Excel 直接双击不乱码
- `dtype=str, keep_default_na=False`：全部按文本读，空字段保留空串而不是 `NaN`


In [ ]:
def save_records(records, filename, columns):
    df = pd.DataFrame(records, columns=columns)
    path = PRIVATE / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def read_records(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)


BASIC_COLUMNS = ["发件人", "邮箱", "主题", "时间"]
basic_path = save_records(basic_records, "mail_basic.csv", BASIC_COLUMNS)
basic_df = read_records(basic_path)
print("CSV记录数：", len(basic_df))
print("列名：", list(basic_df.columns))
basic_df


### 阶段二自查

回读后应满足：**10 行、4 列**。再抽查第一条、中间一条、最后一条与网页逐字段对照，方法见报告「3. 样本核对」。

若邮箱列出现空值，多半是悬停没命中：把 `HOVER_CSS` 改成 `.senders-list` 再试。若出现重复的邮箱，说明第 ① 步的"先移开"没生效，卡片没来得及消失。


## 第三阶段：滚动采集与 ID 去重（第 5—6 课时）

**目标：** 连续采集 3 批，存 5 列 CSV；记录数等于唯一 ID 数；重复采集已有 ID 时新增 0 条。

### 滚动到底是谁在执行

`locator.evaluate()` 是 **Playwright 的方法**，引号里写的是 **JavaScript**，由浏览器执行。Playwright 负责把指令和元素递过去，JS 负责读写滚动容器的属性。

| 属性 | 含义 |
|---|---|
| `scrollTop` | 已向下滚动的距离，可读可写 |
| `clientHeight` | 容器里当前可见的一屏高度 |
| `scrollHeight` | 整个内容的高度 |

下一屏位置是 `top + height`；末尾的最大位置是 `scrollHeight - clientHeight`。两者取较小值，才不会滚过头出现空白。

### 为什么必须按 ID 去重

这不是理论问题，你的 `mail_basic.csv` 里就有现成反例：10 封邮件只有 **5 个发件人名称**，却有 **7 个不同邮箱**（「Microsoft Rewards」同时用了 `customeremail`、`notifyemail`、`infoemail` 三个地址）；还有 **2 组「发件人 + 主题」完全相同**的邮件，只有时间不同。按发件人或主题去重，会把这些真实邮件误删。只有 `data-mailid` 是唯一且稳定的。


### 单元 7：批次解析与滚动循环

`SCROLL_CSS` 必须带 `.mail-list-page-items` 前缀。页面里共有 3 个 `ui-float-scroll-body`（左侧菜单一个、列表一个、另有一个隐藏容器），只写类名会选错。

`seen_ids` 在 `collect_batches` 里创建一次，逐封加入，跨 3 批共用——这是去重能生效的前提。若每批各建一个集合，第 2 批就会把第 1 批的邮件再采一遍。


In [ ]:
SCROLL_CSS = ".mail-list-page-items .ui-float-scroll-body"


def get_box():
    return page.locator(SCROLL_CSS)


async def parse_batch(mail_ids, seen_ids, include_details=False, max_mails=0):
    result = []
    for mail_id in mail_ids:
        if max_mails and len(seen_ids) >= max_mails:
            break
        if mail_id in seen_ids:          # 已在集合里，直接跳过
            continue
        record = await read_list_mail(mail_id)
        if record is None:
            continue
        if include_details:
            # read_detail 在第四阶段定义；本阶段 False，不会执行到这里。
            record.update(await read_detail(mail_id))
        result.append(record)
        seen_ids.add(mail_id)
    return result


async def collect_batches(include_details=False, max_mails=0):
    # 每轮采集前回到收件箱顶部，并新建本轮的去重集合。
    await inbox.click()
    await page.locator(ROW_CSS).first.wait_for()
    await get_box().evaluate("box => box.scrollTop = 0")
    records, seen_ids = [], set()

    for batch in range(3):
        top = await get_box().evaluate("box => box.scrollTop")
        result = await parse_batch(await get_ids(), seen_ids, include_details, max_mails)
        records.extend(result)
        print(f"第{batch + 1}批，新增{len(result)}，累计{len(seen_ids)}")
        if batch == 2 or (max_mails and len(seen_ids) >= max_mails):
            break

        # 定位和悬停都可能触发自动滚动，所以仍按本批开始时记下的 top 计算。
        box = get_box()
        height = await box.evaluate("box => box.clientHeight")
        full_height = await box.evaluate("box => box.scrollHeight")
        next_top = min(top + height, full_height - height)
        if next_top <= top:
            print("已到列表末尾")
            break

        old_ids = await get_ids()
        await box.evaluate("(box, value) => box.scrollTop = value", next_top)
        # 在浏览器里比较新旧 ID 数组，内容真的变了才继续，避免采到同一屏。
        await page.wait_for_function("""data => {
            const ids = Array.from(document.querySelectorAll(data.selector),
                                   node => node.getAttribute('data-mailid'));
            return ids.length > 0 && JSON.stringify(ids) !== JSON.stringify(data.ids);
        }""", arg={"selector": ROW_CSS, "ids": old_ids})
    return records, seen_ids


### 单元 8：执行 3 批并保存

复用单元 6 的 `save_records` / `read_records`，不另写保存函数。

3 批的条数以实际结果为准。虚拟列表每次只渲染约 25 行，滚一屏换一批，批间必然有重叠——重叠部分正是被 `seen_ids` 挡掉的部分，所以「记录数 = 唯一 ID 数」是必然结果，不是巧合。


In [ ]:
PAGE_COLUMNS = ["邮件ID"] + BASIC_COLUMNS
page_records, seen_ids = await collect_batches()
page_path = save_records(page_records, "mail_pages.csv", PAGE_COLUMNS)
page_df = read_records(page_path)
print("CSV记录数：", len(page_df))
print("唯一ID数：", page_df["邮件ID"].nunique())
print("重复ID数：", page_df["邮件ID"].duplicated().sum())


### 单元 9：重放已验证的 ID，证明去重有效

取已经采过的 5 个 ID，用**同一个** `seen_ids` 再跑一次 `parse_batch`，然后重写同一个文件、比较前后字节。

- `parse_batch` 在 `if mail_id in seen_ids` 处就跳过了，根本不会去碰 DOM，所以这些 ID 即使已经滚出屏幕也不影响。
- 新增必须是 **0 条**；文件字节不变，说明内容没被追加。
- 关键在「同一次运行、同一个集合」。如果关掉程序重开、重新覆盖一遍文件，只能证明"文件被重写了"，证明不了去重逻辑能识别重复。


In [ ]:
replay_ids = [r["邮件ID"] for r in page_records[:5]]
assert replay_ids, "先成功采集一些邮件，再做重复验证"

before = page_path.read_bytes()
replayed = await parse_batch(replay_ids, seen_ids)
print("重复采集新增：", len(replayed))

save_records(page_records + replayed, "mail_pages.csv", PAGE_COLUMNS)
print("CSV内容未变：", before == page_path.read_bytes())


### 阶段三完成证据

1. 滚动、`wait_for_function`、停止条件的代码与 3 批真实日志。
2. `CSV记录数`、`唯一ID数`、`重复ID数` 三项截图。
3. `重复采集新增： 0` 与 `CSV内容未变： True` 的截图。

报告「1. 滚动与等待」需要填的答案都在代码注释里：滚动容器选择器 `.mail-list-page-items .ui-float-scroll-body`；每批开始用 `scrollTop` 记录位置；下一屏 `min(top + height, scrollHeight - clientHeight)`；判断更新用 `wait_for_function` 比较新旧 ID 数组。


## 第四阶段：详情正文（第 7—8 课时）

**目标：** 逐封打开详情、读正文、返回列表、继续下一封。正文必做，附件下载选做。

难点在于**状态恢复**：打开详情后，列表会被替换掉，原来那一屏的 DOM 节点全部失效。所以每次返回都必须做两件事——把列表滚回原来的位置，再重新定位那一封邮件。少做一步，后面就会因为"元素找不到"而中断。

这也是为什么 `read_detail` 用 `try/finally`：即使正文读取抛异常，也必须执行返回动作，否则整个浏览器会卡在详情页，后面的邮件一封也采不到。


### 探测单元：导出详情页结构

正文容器和"返回"控件的类名无法凭空猜测，先打开一封看看真实结构。

本单元会自动打开第一封邮件、保存结构、再点回收件箱，所以跑完不会把浏览器留在详情页。


In [ ]:
# 【探测单元】打开第一封邮件，导出详情页结构，然后返回列表。
probe_id = (await get_ids())[0]
probe_row = get_row(probe_id)
await probe_row.locator(SUBJECT_CSS).first.click()
await page.wait_for_timeout(3000)

detail_html = await page.content()
(PRIVATE / "dom_detail.html").write_text(detail_html, encoding="utf-8")
print("详情页 DOM 字符数：", len(detail_html))
print("邮件ID：", probe_id)
# 只打印路径，不打印完整 URL——URL 里带 sid 会话参数，不能进 Notebook 输出。
print("当前页面路径：", page.url.split("?")[0])

# 点回收件箱，保证后续单元还能继续。
await inbox.click()
await page.locator(ROW_CSS).first.wait_for()
print("已返回列表")


### 单元 10：详情正文函数

实测结论：

| 目标 | 取值 | 依据 |
|---|---|---|
| 正文容器 | `.mail-detail-content` | 文本 509 字符，正是邮件正文；其内部是 QQ 邮箱的 `.qmbox` |
| 返回控件 | 文字「返回」 | 全页仅 1 处，位于列表工具栏左端 |

三个必须理解的细节：

1. **`top` 要在点击之前读**。进入详情后列表就被替换了，`scrollTop` 无从读取。所以先记位置，返回后再写回去。
2. **`try/finally` 不是可选项**。正文读取失败时若没有 `finally` 里的返回动作，浏览器会一直停在详情页，后续所有邮件都定位不到。`finally` 保证"无论成败都退出来"。
3. **返回后要重新定位**。详情页里 `data-mailid` 数量为 0，原来的行全部失效。必须重新 `get_row(mail_id)` 等它重新出现，不能沿用旧对象。

正文含换行和大量空白，用 `inner_text()` 取渲染后的文本再 `strip()`。附件开关保持 `False`，选做时再打开。


In [ ]:
BODY_CSS = ".mail-detail-content"
BACK_TEXT = "返回"
DOWNLOAD_ATTACHMENTS = False   # 必做先保持 False；选做时补完单元 13 再开启。


async def read_detail(mail_id):
    # 先记滚动位置：进入详情后列表会被替换，就读不到了。
    top = await get_box().evaluate("box => box.scrollTop")
    row = get_row(mail_id)
    await row.locator(SUBJECT_CSS).first.click()
    try:
        body_node = page.locator(BODY_CSS)
        await body_node.wait_for(state="visible")
        body = (await body_node.inner_text()).strip()
        files = ""
        if DOWNLOAD_ATTACHMENTS:
            # 选做函数在单元 13 定义；开关关闭时不会执行到。
            files = await download_attachments(mail_id)
        return {"正文": body, "附件文件": files}
    finally:
        # 无论正文是否读成功，都必须退回列表，否则后面全部失效。
        await page.get_by_text(BACK_TEXT, exact=True).click()
        await get_box().wait_for()
        await get_box().evaluate("(box, value) => box.scrollTop = value", top)
        await get_row(mail_id).wait_for(state="attached")


### 单元 11：先试 3 封

`include_details=True` 打开正文采集，`max_mails=3` 限 3 封。这两个参数正是本单元要你决定的。

试采会新建自己的去重集合，不会影响后面正式采集的计数。


In [ ]:
DETAIL_COLUMNS = PAGE_COLUMNS + ["正文", "附件文件"]

trial_records, _ = await collect_batches(include_details=True, max_mails=3)
trial_path = save_records(trial_records, "mail_details_trial.csv", DETAIL_COLUMNS)
trial_df = read_records(trial_path)
print(trial_df[["邮件ID", "正文"]].assign(正文长度=trial_df["正文"].str.len()))


### 单元 12：正式采集 3 批正文

去掉 `max_mails`，恢复 3 批全量采集。这是整个实验最慢的一步：每封都要经历"悬停取邮箱 → 进详情 → 读正文 → 返回 → 恢复位置"，预计比第三阶段慢一倍以上，中途不要中断。

试采和正式采集各自新建去重集合，所以两者的条数互不影响。


In [ ]:
detail_records, _ = await collect_batches(include_details=True)
detail_path = save_records(detail_records, "mail_details.csv", DETAIL_COLUMNS)
detail_df = read_records(detail_path)
print("正文CSV记录数：", len(detail_df))
print("重复ID数：", detail_df["邮件ID"].duplicated().sum())


### 阶段四自查

回读后应为 **7 列**：`邮件ID, 发件人, 邮箱, 主题, 时间, 正文, 附件文件`；未下载附件时最后一列为空串。

抽查 3 封，确认正文与详情页可见内容一致，**不是列表里的摘要**。列表摘要来自 `.mail-digest`，正文来自 `.mail-detail-content`，两者来源不同——这是回答思考题 3 的关键。

纯图片邮件正文可能为空或只有占位文字，如实记录即可，不要求 OCR。正文含换行时，用 `len(df)` 统计邮件数，不要数文本行数。
